# Fast Multi-Level Monte Carlo: Results Showcase

**Project:** American Option Pricing via Markovian Projection  
**Author:** Wadoud Charbak (KAUST Intern)  
**Based on:** Amelie's research codebase

---

This notebook demonstrates the refactored Fast Multi-Level (FML) implementation for pricing high-dimensional American basket options. We showcase:

1. **Single-Level MC** - Baseline reference
2. **Multi-Level MC** - Telescoping sum with accumulated normal equations
3. **OT-Enhanced MLMC** - Gaussian-Brenier optimal transport coupling
4. **Hierarchical QR MLMC** - Numerically stable variant

The key innovation is memory efficiency: instead of storing the full design matrix $D \in \mathbb{R}^{MN \times \text{dimV}}$, we accumulate the normal equations $G = D^\top D$ and $g = D^\top \psi$ on-the-fly.

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os

# Create plot directories
os.makedirs('plots/SL', exist_ok=True)
os.makedirs('plots/ML', exist_ok=True)

# Import FML modules
from FML_utils import (
    GBM_paths, scalings_l0, tot_degree_poly, 
    mlmc_level, make_c, make_b_bar
)
from FML_single_level import single_level
from FML_optimal_transport import make_c_OT
from FML_hierarchical_qr import make_c_qr

print("All modules imported successfully!")

In [ ]:
# Basket parameters (3-asset example)
d = 3
P1 = np.ones(d) / d  # Equal-weighted basket
r = 0.05  # Risk-free rate
x0 = np.linspace(225, 275, num=d)[:, np.newaxis]  # Initial prices
vol = np.array([0.2, 0.15, 0.1])  # Volatilities
cov_mat = np.array([[1.0, 0.8, 0.3],
                    [0.8, 1.0, 0.1],
                    [0.3, 0.1, 1.0]])  # Correlation matrix
T = 1.0  # Maturity (1 year)
h0 = 0.01  # Base time step

print(f"Basket Configuration:")
print(f"  Assets: {d}")
print(f"  Initial prices: {x0.flatten()}")
print(f"  Volatilities: {vol}")
print(f"  Maturity: {T} year")

## 2. Single-Level Volatility Surfaces

The single-level Monte Carlo provides our baseline reference. We fit the projected volatility surface $\bar{b}(t, S)$ using orthonormalised Legendre polynomials.

In [ ]:
# Compute scaling parameters
max_deg = 3
s_min0, s_max0, basket0 = scalings_l0(
    x0, T, h0, r, cov_mat, vol, max_deg, P1, M_0=10000, return_basket=True
)
print(f"Basket scaling range: [{s_min0:.2f}, {s_max0:.2f}]")

# Visualisation bounds
s_min_vis = np.percentile(basket0, 1, axis=0)
s_max_vis = np.percentile(basket0, 99, axis=0)

# Time grid for plotting
dt = h0 * 2 ** (-max_deg)
N_t = int(T / dt)
t = np.linspace(0, T, N_t)

In [ ]:
# Compute Single-Level coefficients
print("Computing Single-Level volatility surface...")
pairs = tot_degree_poly(max_deg)
c_SL = single_level(x0, T, h0, r, cov_mat, vol, max_deg, P1, s_min0, s_max0, C=80)
bbar_SL = make_b_bar(c_SL, pairs, s_min0, s_max0, T, max_deg)
print(f"Fitted {len(pairs)} basis functions")

In [ ]:
def plot_surface(bbar, s_min_vis, s_max_vis, t, title, cmap='viridis'):
    """Helper function to plot volatility surface."""
    K, L = 50, 150
    t_1 = np.linspace(0, t[-1], K)
    idx = np.searchsorted(t, t_1)
    
    TT = np.zeros((L, K))
    SS = np.zeros((L, K))
    for j, ti in enumerate(idx):
        TT[:, j] = t_1[j]
        SS[:, j] = np.linspace(s_min_vis[ti], s_max_vis[ti], L)
    
    bbar_vals = bbar(TT, SS)
    
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(TT, SS, bbar_vals, cmap=cmap, rcount=40, ccount=40, alpha=0.9)
    ax.set_xlabel('Time $t$', fontsize=12)
    ax.set_ylabel('Basket $S$', fontsize=12)
    ax.set_zlabel(r'$\bar{b}(t,S)$', fontsize=12)
    ax.set_title(title, fontsize=14)
    fig.colorbar(surf, shrink=0.5, aspect=10)
    return fig, ax, bbar_vals

# Plot Single-Level surface
fig, ax, bbar_SL_vals = plot_surface(
    bbar_SL, s_min_vis, s_max_vis, t,
    f'Single-Level Volatility Surface (degree = {max_deg})'
)
plt.savefig('plots/SL/VolSurf_maxdeg3.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 3. Multi-Level Volatility Surfaces

The MLMC approach uses the telescoping sum:
$$c = \sum_{l=0}^{L} c_l$$

where each $c_l$ is computed at level $l$ with polynomial degree $L - l$ (multi-resolution principle).

In [ ]:
# Compute Multi-Level coefficients
print("Computing Multi-Level volatility surface...")
c_ML = make_c(x0, T, h0, r, cov_mat, vol, max_deg, P1, s_min0, s_max0, C=80)
bbar_ML = make_b_bar(c_ML, pairs, s_min0, s_max0, T, max_deg)
print("Done!")

In [ ]:
# Plot Multi-Level surface
fig, ax, bbar_ML_vals = plot_surface(
    bbar_ML, s_min_vis, s_max_vis, t,
    f'Multi-Level Volatility Surface (degree = {max_deg})',
    cmap='plasma'
)
plt.savefig('plots/ML/VolSurf_maxdeg3.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 4. Optimal Transport Enhanced MLMC

The OT-MLMC uses Gaussian-Brenier maps to achieve optimal coupling between fine and coarse paths. For Gaussian distributions (which log-GBM produces), the optimal map is:

$$T(x) = \mu_c + A(x - \mu_f)$$

where $A = C_f^{-1/2} (C_f^{1/2} C_c C_f^{1/2})^{1/2} C_f^{-1/2}$

In [ ]:
# Compute OT-Enhanced coefficients
print("Computing OT-Enhanced Multi-Level volatility surface...")
c_OT = make_c_OT(x0, T, h0, r, cov_mat, vol, max_deg, P1, s_min0, s_max0, C=80)
bbar_OT = make_b_bar(c_OT, pairs, s_min0, s_max0, T, max_deg)
print("Done!")

In [ ]:
# Plot OT surface
fig, ax, bbar_OT_vals = plot_surface(
    bbar_OT, s_min_vis, s_max_vis, t,
    f'OT-Enhanced MLMC Volatility Surface (degree = {max_deg})',
    cmap='inferno'
)
plt.savefig('plots/ML/VolSurf_OT_maxdeg3.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hierarchical QR MLMC

The hierarchical QR approach uses three-level QR decomposition for improved numerical stability. Unlike the normal equations approach (which squares the condition number), QR preserves the original conditioning.

In [ ]:
# Compute Hierarchical QR coefficients
print("Computing Hierarchical QR Multi-Level volatility surface...")
c_QR = make_c_qr(x0, T, h0, r, cov_mat, vol, max_deg, P1, s_min0, s_max0, C=80)
bbar_QR = make_b_bar(c_QR, pairs, s_min0, s_max0, T, max_deg)
print("Done!")

In [ ]:
# Plot QR surface
fig, ax, bbar_QR_vals = plot_surface(
    bbar_QR, s_min_vis, s_max_vis, t,
    f'Hierarchical QR MLMC Volatility Surface (degree = {max_deg})',
    cmap='cividis'
)
plt.savefig('plots/ML/VolSurf_QR_maxdeg3.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 6. Side-by-Side Comparison

Let's compare all four methods in a single figure.

In [ ]:
# Create 2x2 comparison plot
fig = plt.figure(figsize=(16, 14))

# Prepare grid
K, L = 50, 150
t_1 = np.linspace(0, t[-1], K)
idx = np.searchsorted(t, t_1)

TT = np.zeros((L, K))
SS = np.zeros((L, K))
for j, ti in enumerate(idx):
    TT[:, j] = t_1[j]
    SS[:, j] = np.linspace(s_min_vis[ti], s_max_vis[ti], L)

methods = [
    ('Single-Level', bbar_SL, 'viridis'),
    ('Multi-Level', bbar_ML, 'plasma'),
    ('OT-Enhanced MLMC', bbar_OT, 'inferno'),
    ('Hierarchical QR MLMC', bbar_QR, 'cividis')
]

for i, (name, bbar, cmap) in enumerate(methods):
    ax = fig.add_subplot(2, 2, i+1, projection='3d')
    vals = bbar(TT, SS)
    ax.plot_surface(TT, SS, vals, cmap=cmap, rcount=30, ccount=30, alpha=0.9)
    ax.set_xlabel('Time $t$')
    ax.set_ylabel('Basket $S$')
    ax.set_zlabel(r'$\bar{b}(t,S)$')
    ax.set_title(name, fontsize=14)

plt.suptitle(f'Volatility Surface Comparison (degree = {max_deg})', fontsize=16, y=0.98)
plt.tight_layout()
plt.savefig('plots/VolSurf_Comparison.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 7. Difference Plots

Visualise the difference between methods to verify consistency.

In [ ]:
# Compute surfaces on common grid
val_SL = bbar_SL(TT, SS)
val_ML = bbar_ML(TT, SS)
val_OT = bbar_OT(TT, SS)
val_QR = bbar_QR(TT, SS)

# Difference plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

diffs = [
    ('SL - ML', val_SL - val_ML),
    ('SL - OT', val_SL - val_OT),
    ('SL - QR', val_SL - val_QR)
]

for ax, (name, diff) in zip(axes, diffs):
    im = ax.imshow(diff, aspect='auto', origin='lower', cmap='RdBu_r',
                   extent=[0, T, s_min_vis.min(), s_max_vis.max()])
    ax.set_xlabel('Time $t$')
    ax.set_ylabel('Basket $S$')
    ax.set_title(f'{name}\nMax |diff| = {np.abs(diff).max():.4e}')
    plt.colorbar(im, ax=ax)

plt.suptitle('Difference Between Methods', fontsize=14)
plt.tight_layout()
plt.savefig('plots/Method_Differences.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary Statistics

In [ ]:
print("="*60)
print("Summary Statistics")
print("="*60)
print(f"\nPolynomial degree: {max_deg}")
print(f"Number of basis functions: {len(pairs)}")
print(f"\nVolatility Surface Statistics:")
print(f"  SL:  mean = {val_SL.mean():.4f}, std = {val_SL.std():.4f}, range = [{val_SL.min():.4f}, {val_SL.max():.4f}]")
print(f"  ML:  mean = {val_ML.mean():.4f}, std = {val_ML.std():.4f}, range = [{val_ML.min():.4f}, {val_ML.max():.4f}]")
print(f"  OT:  mean = {val_OT.mean():.4f}, std = {val_OT.std():.4f}, range = [{val_OT.min():.4f}, {val_OT.max():.4f}]")
print(f"  QR:  mean = {val_QR.mean():.4f}, std = {val_QR.std():.4f}, range = [{val_QR.min():.4f}, {val_QR.max():.4f}]")
print(f"\nMaximum Absolute Differences from Single-Level:")
print(f"  |SL - ML|_max = {np.abs(val_SL - val_ML).max():.4e}")
print(f"  |SL - OT|_max = {np.abs(val_SL - val_OT).max():.4e}")
print(f"  |SL - QR|_max = {np.abs(val_SL - val_QR).max():.4e}")

## 9. Coefficient Comparison

In [ ]:
# Compare coefficients
fig, ax = plt.subplots(figsize=(12, 5))

x_pos = np.arange(len(pairs))
width = 0.2

ax.bar(x_pos - 1.5*width, c_SL, width, label='Single-Level', alpha=0.8)
ax.bar(x_pos - 0.5*width, c_ML, width, label='Multi-Level', alpha=0.8)
ax.bar(x_pos + 0.5*width, c_OT, width, label='OT-MLMC', alpha=0.8)
ax.bar(x_pos + 1.5*width, c_QR, width, label='QR-MLMC', alpha=0.8)

ax.set_xlabel('Basis Function Index')
ax.set_ylabel('Coefficient Value')
ax.set_title('Comparison of Fitted Coefficients')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'({i},{j})' for i,j in pairs], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/Coefficient_Comparison.pdf', dpi=150, bbox_inches='tight')
plt.show()

---

## Conclusion

All four methods produce consistent volatility surfaces, validating the refactored implementation. Key observations:

1. **Single-Level** provides the reference baseline
2. **Multi-Level** matches SL while using the memory-efficient accumulated normal equations
3. **OT-Enhanced** achieves optimal fine-coarse coupling via Gaussian-Brenier maps
4. **Hierarchical QR** offers superior numerical stability

For detailed error analysis and statistical comparisons, see `FML_Method_Comparison.ipynb`.